# Data Cleaning & Mapping — Assessment PEREN AI


Objectifs :
- Nettoyer et structurer les données issues du formulaire d’assessment
- Normaliser les variables catégorielles
- Créer des indicateurs **neutres** (sans interprétation métier)
- Poser les bases du **data model MVP**

⚠️ Aucun score santé ou modèle prédictif n’est calculé à ce stade.


In [4]:
import pandas as pd
import numpy as np


Librairies utilisées :
- pandas : manipulation et transformation des données tabulaires
- numpy : gestion des valeurs manquantes et calculs numériques simples


In [5]:
#load data
df_raw = pd.read_csv("../data/processed/01_Dataset_clean_merge.csv")
df = df_raw.copy()


Les colonnes sont renommées selon les conventions suivantes :
- snake_case
- noms explicites
- unités incluses lorsque nécessaire (cm, kg)

Cela garantit :
- lisibilité
- compatibilité avec les futures pipelines data / ML


In [6]:
df["sex"] = df["sex"].map({"F": "female", "H": "male"})


La variable `sex` est normalisée pour éviter toute ambiguïté
et faciliter l’exploitation future (modèles, règles métier).


In [7]:
#IMC
df["height_m"] = df["height_cm"] / 100
df["bmi"] = df["weight_kg"] / (df["height_m"] ** 2)


In [8]:
#stress
stress_map = {
    "Faible": "low",
    "Modéré": "moderate",
    "Élevé": "high"
}

df["stress_level_norm"] = df["stress_level"].map(stress_map)


In [9]:
#sommeil
df["sleep_6h_plus_norm"] = df["sleep_duration"].apply(lambda x: 0 if x == ">6h" else 1)

In [10]:
#Nutrition
nutrition_map = {
    "Maison": "equilibrated",
    "Mix maison": "mixed",
    "Mix": "mixed",
    "Équilibrée": "equilibrated"
}

df["nutrition_norm"] = df["nutrition_raw"].map(nutrition_map)



Mapping validé par l’équipe métier :

- "Maison" → Équilibrée
- "Mix maison" → Mixte

Ce mapping est documenté pour assurer la traçabilité des décisions.


In [11]:
df["family_history_flag"] = df["family_history_raw"].apply(lambda x: 0 if x == "Aucun" else 1)


In [12]:
df.isnull().mean() * 100


user_id                 0.000000
datetime                0.000000
sex                    44.916667
age                     0.000000
height_cm               0.000000
weight_kg               0.000000
Sport_type              0.000000
activity_freq           0.000000
sleep_duration          0.000000
stress_level            0.000000
nutrition_raw           0.000000
sedentary_time          0.000000
alcohol_raw             0.000000
family_history_raw      0.000000
cycle_raw               0.000000
date                    0.000000
time                    0.000000
height_m                0.000000
bmi                     0.000000
stress_level_norm       0.000000
sleep_6h_plus_norm      0.000000
nutrition_norm         44.583333
family_history_flag     0.000000
dtype: float64

In [13]:
df.to_csv("../data/processed/2_PerenAI_dataset_clean.csv", index=False)


In [14]:
df.head()

,user_id,datetime,sex,age,height_cm,weight_kg,Sport_type,activity_freq,sleep_duration,stress_level,...,family_history_raw,cycle_raw,date,time,height_m,bmi,stress_level_norm,sleep_6h_plus_norm,nutrition_norm,family_history_flag
0,U01,2025-10-18 08:12:34,female,32,165,58,Running,≥5,7–8h,Modéré,...,Aucun,Régulier,2025-10-18 00:00:00,08:12:34,1.65,21.303949,moderate,1,equilibrated,0
1,U01,2025-11-02 07:58:21,female,32,165,59,Running,≥5,7–8h,Modéré,...,Aucun,Régulier,2025-11-02 00:00:00,07:58:21,1.65,21.671258,moderate,1,equilibrated,0
2,U01,2025-11-15 08:20:41,female,32,165,58,Running,≥5,7–8h,Modéré,...,Aucun,Régulier,2025-11-15 00:00:00,08:20:41,1.65,21.303949,moderate,1,equilibrated,0
3,U02,2025-10-20 10:45:12,male,38,178,75,CrossFit,4,6–7h,Élevé,...,HTA,—,2025-10-20 00:00:00,10:45:12,1.78,23.671254,high,1,mixed,1
4,U02,2025-11-04 09:40:08,male,38,178,76,CrossFit,4,6–7h,Modéré,...,HTA,—,2025-11-04 00:00:00,09:40:08,1.78,23.986870,moderate,1,mixed,1
